# Pilot Analysis

Pilot-stage analysis for the Bug Report Quality Assessment with LLM project.

This notebook reads pilot result files and prints:
- summary metrics
- descriptive statistics
- text-based bars
- prediction label distribution


## 1. Setup

In [ ]:
from pathlib import Path
import csv
import math

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "Data").exists() and (CURRENT_DIR / "Results").exists():
    PROJECT_ROOT = CURRENT_DIR
elif CURRENT_DIR.name.lower() == "results" and (CURRENT_DIR.parent / "Data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

KAPPA_THRESHOLD = 0.70

print("Current dir :", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)

RAW_RESULTS_DIR = PROJECT_ROOT / "Results" / "Raw"
IMPROVED_RESULTS_DIR = PROJECT_ROOT / "Results" / "Improved"

RAW_SUMMARY = RAW_RESULTS_DIR / "summary_raw.csv"
IMPROVED_SUMMARY = IMPROVED_RESULTS_DIR / "summary_improved.csv"

RAW_MISMATCH = RAW_RESULTS_DIR / "mismatch_analysis_raw.csv"
IMPROVED_MISMATCH = IMPROVED_RESULTS_DIR / "mismatch_analysis_improved.csv"

RAW_LLM_OUTPUT = RAW_RESULTS_DIR / "pilot_llm_output_raw.csv"
IMPROVED_LLM_OUTPUT = IMPROVED_RESULTS_DIR / "pilot_llm_output_improved.csv"


## 2. Helper Functions

In [ ]:
def read_csv_rows(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def read_csv_raw_rows(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.reader(file))


def count_data_rows(path):
    rows = read_csv_raw_rows(path)
    return max(len(rows) - 1, 0)


def read_summary(path):
    rows = read_csv_raw_rows(path)

    if not rows:
        raise ValueError(f"Empty summary file: {path}")

    if len(rows[0]) == 2 and len(rows) > 1:
        return {
            row[0].strip(): row[1].strip()
            for row in rows
            if len(row) >= 2 and row[0].strip()
        }

    if len(rows) >= 2:
        header = [col.strip() for col in rows[0]]
        values = rows[1]

        return {
            header[i]: values[i].strip()
            for i in range(min(len(header), len(values)))
        }

    raise ValueError(f"Cannot parse summary file: {path}")


def get_value(summary, possible_keys, default=None):
    normalized = {
        str(key).strip().lower(): value
        for key, value in summary.items()
    }

    for key in possible_keys:
        clean_key = key.strip().lower()

        if clean_key in normalized:
            return normalized[clean_key]

    return default


def to_float(value):
    if value is None:
        return None

    try:
        return float(value)
    except ValueError:
        return None


def normalize_issue_key(value):
    if value is None:
        return ""

    return (
        str(value)
        .strip()
        .replace(" Raw", "")
        .replace(" Improved", "")
        .replace("_raw", "")
        .replace("_improved", "")
        .strip()
    )


def normalize_label(value):
    if value is None:
        return ""

    text = str(value).strip().lower().replace("_", "-")

    if text in {"executable", "exec"}:
        return "Executable"

    if text in {
        "non-executable",
        "non executable",
        "nonexec",
        "non-exec",
        "not executable",
        "not-executable",
    }:
        return "Non-Executable"

    return str(value).strip()


def find_column(row, candidates):
    lower_map = {
        column.strip().lower(): column
        for column in row.keys()
    }

    for candidate in candidates:
        clean_candidate = candidate.strip().lower()

        if clean_candidate in lower_map:
            return lower_map[clean_candidate]

    return None


def print_table(rows, columns):
    if not rows:
        print("(empty)")
        return

    widths = {}

    for column in columns:
        widths[column] = max(
            len(str(column)),
            max(len(str(row.get(column, ""))) for row in rows),
        )

    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)

    print(header)
    print(separator)

    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))


def ascii_bar(label, value, max_value=1.0, width=40):
    if value is None:
        print(f"{label}: N/A")
        return

    filled = int((value / max_value) * width) if max_value else 0
    filled = max(0, min(width, filled))
    bar = "#" * filled + "-" * (width - filled)

    print(f"{label:<10} |{bar}| {value:.4f}")


def describe_values(values):
    clean = [value for value in values if value is not None]

    if not clean:
        return {
            "count": 0,
            "min": None,
            "max": None,
            "mean": None,
        }

    return {
        "count": len(clean),
        "min": round(min(clean), 4),
        "max": round(max(clean), 4),
        "mean": round(sum(clean) / len(clean), 4),
    }


def exact_mcnemar_p_value(b, c):
    n = b + c

    if n == 0:
        return 1.0

    k = min(b, c)
    p_value = 2 * sum(math.comb(n, i) * (0.5 ** n) for i in range(0, k + 1))

    return min(p_value, 1.0)


## 3. Load Pilot Results

In [ ]:
raw_summary = read_summary(RAW_SUMMARY)
improved_summary = read_summary(IMPROVED_SUMMARY)

raw_mismatch_cases = count_data_rows(RAW_MISMATCH)
improved_mismatch_cases = count_data_rows(IMPROVED_MISMATCH)

pilot_metrics = [
    {
        "version": "Raw",
        "accuracy": to_float(get_value(raw_summary, ["Accuracy", "accuracy"])),
        "cohen_kappa": to_float(get_value(raw_summary, ["Cohen's Kappa", "Cohen Kappa", "cohen_kappa"])),
        "threshold": KAPPA_THRESHOLD,
        "threshold_passed": str(get_value(raw_summary, ["Threshold passed", "threshold_passed"], "")).lower() == "true",
        "mismatch_cases": raw_mismatch_cases,
    },
    {
        "version": "Improved",
        "accuracy": to_float(get_value(improved_summary, ["Accuracy", "accuracy"])),
        "cohen_kappa": to_float(get_value(improved_summary, ["Cohen's Kappa", "Cohen Kappa", "cohen_kappa"])),
        "threshold": KAPPA_THRESHOLD,
        "threshold_passed": str(get_value(improved_summary, ["Threshold passed", "threshold_passed"], "")).lower() == "true",
        "mismatch_cases": improved_mismatch_cases,
    },
]

print_table(
    pilot_metrics,
    ["version", "accuracy", "cohen_kappa", "threshold", "threshold_passed", "mismatch_cases"],
)


## 4. Descriptive Statistics

In [ ]:
stats_rows = [
    {"metric": "accuracy", **describe_values([row["accuracy"] for row in pilot_metrics])},
    {"metric": "cohen_kappa", **describe_values([row["cohen_kappa"] for row in pilot_metrics])},
    {"metric": "mismatch_cases", **describe_values([row["mismatch_cases"] for row in pilot_metrics])},
]

print_table(stats_rows, ["metric", "count", "min", "max", "mean"])


## 5. Text-based Bars

In [ ]:
print("Accuracy")
for row in pilot_metrics:
    ascii_bar(row["version"], row["accuracy"], max_value=1.0)

print()
print("Cohen's Kappa")
for row in pilot_metrics:
    ascii_bar(row["version"], row["cohen_kappa"], max_value=1.0)

print()
print("Mismatch cases")
max_mismatch = max(row["mismatch_cases"] for row in pilot_metrics)

for row in pilot_metrics:
    ascii_bar(row["version"], row["mismatch_cases"], max_value=max_mismatch)


## 6. Prediction Label Distribution

In [ ]:
def label_distribution(path):
    rows = read_csv_rows(path)

    if not rows:
        return {}

    label_col = find_column(rows[0], ["s2r_label", "S2R Label", "prediction", "llm_prediction"])

    if not label_col:
        return {}

    counts = {}

    for row in rows:
        label = normalize_label(row.get(label_col, ""))
        counts[label] = counts.get(label, 0) + 1

    return counts


raw_label_dist = label_distribution(RAW_LLM_OUTPUT)
improved_label_dist = label_distribution(IMPROVED_LLM_OUTPUT)

label_rows = []

for label, count in raw_label_dist.items():
    label_rows.append({"version": "Raw", "label": label, "count": count})

for label, count in improved_label_dist.items():
    label_rows.append({"version": "Improved", "label": label, "count": count})

print_table(label_rows, ["version", "label", "count"])


## 7. Test Choice Note

For pilot data, the sample size is small, so descriptive statistics are the safest focus.
For full paired Raw vs Improved correctness comparison, use McNemar exact test.
